# 6 自定义量化 A8W8 matmul 算子开发并接入 Qwen3-8B

本章是整个启航营的**核心实践环节**，将学习到的所有知识沉淀转化为一次完整的自定义算子开发实战。

实践的环境准备、提交方式可参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

## 本章定位

本章将开发一个自定义的量化 matmul 算子 `QmmCustom`，通过 A8W8 量化（Activation INT8 + Weight INT8）减少计算量和内存占用，从而提升推理性能。这是本次启航营中综合性最强的一章——涵盖了从算子概念理解、Tiling 设计、Kernel 实现到算子编译、接入网络与功能性能测试的全链路。

## 通过实践达成的学习目标

- **理解量化推理原理**：掌握 A8W8 量化方案（INT8 激活 + INT8 权重）的数学原理与 Scale 反量化机制，理解 perChannelScale 与 perTokenScale 的作用。
- **设计 Tiling 数据结构与函数**：根据算子原型定义自行设计 TilingData 结构体，实现多核分块的 Tiling 函数，理解 Cube 计算单元的分块策略。
- **实现 Cube+Vector 双路径 Kernel**：完成 Cube-only（INT32 输出）和 Cube+Vector 协作（反量化 BF16 输出）两条 Kernel 路径的实现，理解昇腾 AI Core 的 Cube 与 Vector 协作编程模型。
- **完成算子编译与 Torch 接入**：编译自定义算子并通过 Torch 接口接入，打通“算子开发 → 框架调用”的链路。
- **测试算子功能与性能，尝试优化迭代**：验证自定义算子的精度正确性，测试单算子与网络性能，并根据测试结果对 Tiling 或 Kernel 进行优化迭代。

## 本章大纲

- 环境准备
- A8W8 量化 matmul 算子概念
- 自定义算子实现
    - **任务一：自行设计tilingdata结构体并实现tiling函数**
    - **任务二：实现A8W8输入，INT32输出的kernel**
    - **任务三：实现A8W8输入，经过perChannelScale和perTokenScale反量化输出BF16的kernel**
- 编译自定义算子
    - **任务四：成功编译自定义算子**
- 测试自定义算子功能性能
    - **任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程**
- 自定义算子接入模型
- 模型性能测试
    - **任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程**
- 提交要求
---

## 1 环境准备

定位教程目录、准备运行目录，并把 CANN 环境变量导入当前 Notebook 进程：

In [1]:
import json
import os
import subprocess
import sys
from pathlib import Path

print('[环境准备] 开始', flush=True)

def locate_repo_root():
    repo_roots = []
    try:
        cwd = Path.cwd()
        lineage = [cwd, *cwd.parents]
        repo_roots.extend(lineage)
        repo_roots.extend(base / 'cann-learning-hub' for base in lineage)
        for base in lineage:
            try:
                repo_roots.extend(base.glob('*/cann-learning-hub'))
            except OSError:
                pass
    except FileNotFoundError:
        pass
    repo_roots.append(Path('/opt/atomgit/cann-learning-hub'))

    seen = set()
    for repo_root in repo_roots:
        key = str(repo_root)
        if key in seen:
            continue
        seen.add(key)
        if (repo_root / 'tutorials/llm_inference/qwen3_8b/src').exists():
            return repo_root
    raise FileNotFoundError('Cannot locate cann-learning-hub repository root.')

REPO_ROOT = locate_repo_root()
os.chdir(REPO_ROOT)
TUTORIAL_DIR = REPO_ROOT / 'tutorials' / 'llm_inference' / 'qwen3_8b'
TUTORIAL_SRC_DIR = TUTORIAL_DIR / 'src'
if str(TUTORIAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(TUTORIAL_SRC_DIR))
from inference_scripts.environment_check import require_unified_environment
require_unified_environment(TUTORIAL_DIR / 'requirements.txt')
SRC_DIR = TUTORIAL_DIR / 'src' / 'op_custom'

if not os.environ.get('ASCEND_TOOLKIT_HOME'):
    raise EnvironmentError('ASCEND_TOOLKIT_HOME is not set. Please activate the CANN environment.')
print('CANN包路径 =', os.environ.get('ASCEND_TOOLKIT_HOME'))
cann_script = Path(os.environ['ASCEND_TOOLKIT_HOME']) / 'set_env.sh'
env_cmd = f'source {cann_script} && env'
env = subprocess.check_output(f"bash -lc '{env_cmd}'", shell=True, text=True, cwd=REPO_ROOT)
for line in env.splitlines():
    if '=' in line:
        os.environ.__setitem__(*line.split('=', 1))

RECIPE_ROOT = TUTORIAL_DIR / 'src/inference_scripts/recipe_qwen3_8b'
QMM_CUSTOM_DIR = SRC_DIR / 'qmm_custom'

print('教程目录 =', TUTORIAL_DIR)
print('推理模型目录 =', RECIPE_ROOT)
print('算子相关文件目录 =', SRC_DIR)
print('算子实现目录 =', QMM_CUSTOM_DIR)
print('[环境准备] 完成', flush=True)

[环境准备] 开始


[依赖检查] 统一环境已就绪，复用第 2 章环境
CANN包路径 = /home/developer/Ascend/cann-9.0.0


教程目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b
推理模型目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b
算子相关文件目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom
算子实现目录 = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom
[环境准备] 完成


本章直接复用第 2 章检查完成的统一环境，不再执行依赖安装。

---
## 2 A8W8 量化 matmul 算子概念

### 2.1 什么是 A8W8 量化

A8W8 量化是 LLM 推理中常用的一种量化方案：
- **A（Activation）**：将输入激活值从 BF16/FP16 量化为 **INT8**
- **W（Weight）**：将权重从 BF16/FP16 量化为 **INT8**
- **反量化**：计算结果 `INT8 × INT8 → INT32` 后，通过 Scale 反量化回浮点精度

> **术语说明**：A8W8 与第 5 章使用的 W8A8 是同一种 INT8 激活 + INT8 权重量化方案的不同写法，本章沿用 A8W8 命名。仓库中 YAML 配置文件名（如 `qwen3_8b_a8w8_1tp.yaml`）和类名（`CompressedTensorsW8A8Int8LinearMethod`）也分别使用了这两种写法。

### 2.2 算子规格

在上一章中，通过分析量化模型的profiling，归纳出了自定义算子的输入输出规格：

| 算子名称 | QmmCustom |
|------|-----|
| 输入 x1 (A) | INT8, ND 格式, shape [M, K] |
| 输入 x2 (B) | INT8, **FRACTAL_NZ** 格式, shape [K, N] |
| 输入 scale | FLOAT32, ND 格式, shape [N] |
| 输入 pertoken_scale | FLOAT32, ND 格式, shape [M]（可选） |
| 输出 y | 有 perTokenScale → BF16，无 → INT32；ND 格式, shape [M, N] |
| 架构 | ascend910b |

**注意**：x2 权重矩阵使用 **FRACTAL_NZ** 格式，这是昇腾 AI Core Cube 单元的高效存储格式，能提升 matmul 计算性能。

### 2.3 Scale 模式

QmmCustom 算子支持两种 Scale：

| Scale 类型 | Shape | 作用 | 
|-----------|-------|------|
| **perChannelScale** | `[N]` | 对每个输出通道单独缩放 |
| **perTokenScale** | `[M]` | 对每行输入单独缩放 |

计算公式：
```
result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale
```

## 3 自定义算子实现

与前面学习过的内容一致，本次实践采用 `<<<>>>` Kernel 直调方式开发上述原型的量化 matmul 算子。直调方式通过 `kernel<<<numBlocks, dynUBufSize, stream>>>(args)` 语法在 Host 侧直接启动 Kernel，适合快速验证算子功能与性能。

本节包含如下三个任务：

### 任务一：自行设计 TilingData 结构体并实现 Tiling 函数

根据算子原型与 Kernel 的实现需求，自行设计 `QmmCustomTilingData` 结构体，并实现 `CalcQmmTiling` 函数。Tiling 函数需根据输入的 M、N、K 维度，结合平台硬件资源（L1/L0A/L0B/UB 内存限制、可用核数），计算多核分块参数。可借助 `platform_ascendc::PlatformAscendCManager` 获取硬件信息。

> **Tiling 设计提示**：
> 
> 设计 TilingData 时需要考虑以下要素：
> 1. **`TCubeTiling`**：CANN Matmul 库提供的标准 Tiling 结构体，包含 M/N/K 的分块大小（`baseM`、`baseN`、`baseK`）、使用的核数（`usedCoreNum`）等字段。可通过 `GetMatmulTiling` API 自动计算。
> 2. **`isPertoken`**：标记是否走反量化路径，用于 Kernel 内部分支选择。
> 3. **`workspaceSize`**：Path 2 中 Cube 写入 INT32 中间结果所需的 GM workspace 大小。
> 4. **多核分块策略**：将 M 维度或 N 维度分配到多个核上并行计算，每个核处理一个分块。分块大小需平衡 L1/L0A/L0B 缓存利用率与多核负载均衡。

### 任务二：实现 A8W8 输入、INT32 输出的 Kernel

实现 `QmmCubeBasicKernel` 类，完成纯 Cube 路径的 INT8 矩阵乘法计算。

计算公式： ```result_int32 = x1_int8 @ x2_int8```

### 任务三：实现 A8W8 输入、经 perChannelScale 和 perTokenScale 反量化输出 BF16 的 Kernel

实现 `QmmPertokenKernel` 类，在 Cube 计算的基础上增加 Vector 反量化路径。

计算公式： ```result_bf16 = (x1_int8 @ x2_int8) * perChannelScale * perTokenScale```

其中 `perChannelScale` 沿 N 维广播，`perTokenScale` 沿 M 维广播，两者均为 FLOAT32 类型。

---

### 执行路径说明

在 Host 侧的 `qmm_custom` 函数中，根据是否传入 `pertoken_scale` 设置 `isPertoken` 标志，Kernel 启动时据此选择执行路径：

| 路径 | 条件 | 计算流程 | 输出类型 |
|------|------|----------|----------|
| **Path 1: Cube-only** | `isPertoken == 0` | INT8 matmul → INT32 | INT32 |
| **Path 2: Cube+Vector** | `isPertoken == 1` | INT8 matmul → INT32 → dequant(scale×pertoken) → BF16 | BF16 |

**Path 1（Cube-only）**：仅使用 AI Core 的 Cube 计算单元，完成 INT8×INT8 矩阵乘法后直接将 INT32 结果写入输出 GM，无 Vector 参与。对应 `QmmCubeBasicKernel` 的实现。

**Path 2（Cube+Vector）**：Cube 单元先完成 INT8 matmul 得到 INT32 中间结果并写入 GM workspace；随后 Vector 单元从 GM 读取 INT32 结果，依次完成 `Cast`（INT32→FLOAT32）、乘以 `perChannelScale` 和 `perTokenScale`、`Cast`（FLOAT32→BF16）等反量化操作，最终将 BF16 结果写入输出 GM。对应 `QmmPertokenKernel` 的实现。

Kernel 入口函数 `qmm_custom_kernel` 使用 `__mix__(1, 2)` 属性声明，表示为每个 Block 分配 1 个 Cube 核和 2 个 Vector 核，使 Cube 与 Vector 可在同一 Kernel 中协同工作。Kernel 内部通过 `tilingData.isPertoken` 判断走哪条路径，两条路径复用相同的 Tiling 结构和 Kernel 启动入口。

Tiling、Kernel、Torch 接口的实现均位于 `qmm_custom.asc` 文件中。文件已给出整体框架与函数签名，请自行完成下方代码中的 TODO 部分，以实现完整的自定义算子：

In [2]:
%%writefile {QMM_CUSTOM_DIR}/qmm_custom.asc
#include "acl/acl_base.h"
#include "acl/acl_prof.h"
#include "kernel_operator.h"
#include "lib/matmul_intf.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
#include "torch_npu/csrc/core/npu/NPUStream.h"
#include <torch/extension.h>

// 任务一: 自行设计tilingdata结构体并实现tiling函数
// ---- 反量化库（任务三 perChannel/perToken 反量化使用）----
#if __has_include("quantization/ascend_dequant.h")
#include "quantization/ascend_dequant.h"
#elif __has_include("adv_api/quantization/ascend_dequant.h")
#include "adv_api/quantization/ascend_dequant.h"
#endif
#include <array>
#include <map>

using namespace AscendC;

// ---- kernel/host 侧共用的设计常量 ----
// FRACTAL_NZ 的列分形长度 n0：N 方向核间切分必须按它对齐，才能换算成 B 矩阵的 NZ 物理偏移
constexpr uint32_t QMM_NZ_N0 = 32;
// AIV 反量化 N 方向条带宽度与 UB 预算（元素数，按每元素约 20B 的双缓冲开销折算）
constexpr uint32_t QMM_VEC_STRIP_N = 1024;
constexpr uint32_t QMM_VEC_UB_BUDGET_ELEMS = 7168;
// AIC <-> AIV 跨核同步通道（与量产混核算子 grouped_matmul 的用法保持一致）
constexpr uint16_t QMM_FLAG_AIC2AIV = 5;
constexpr uint16_t QMM_FLAG_AIV2AIC = 3;
constexpr uint8_t QMM_SYNC_MODE = 2;

// C 结果直接经 fixpipe 写 GM、Cube-Vector 同步由 CrossCoreSetFlag/WaitFlag 手工完成，
// 因此必须关闭 CFG_NORM 默认的 enUnitFlag（开启后 Matmul 库会在 AIC 上访问不存在的 UB）
constexpr MatmulConfig QMM_MM_CFG = GetNormalConfig(
    false, false, false, BatchMode::BATCH_LESS_THAN_L1, true, IterateOrder::UNDEF,
    ScheduleType::INNER_PRODUCT, false, false);

// A(int8 ND) x B(int8 FRACTAL_NZ) -> C(int32 ND)：INT8 相乘在 L0C 用 int32 精确累加，无浮点误差
using QmmAType = MatmulType<TPosition::GM, CubeFormat::ND, int8_t>;
using QmmBType = MatmulType<TPosition::GM, CubeFormat::NZ, int8_t>;
using QmmCType = MatmulType<TPosition::GM, CubeFormat::ND, int32_t>;
using QmmMatmul = MatmulImpl<QmmAType, QmmBType, QmmCType, QmmCType, QMM_MM_CFG>;

__aicore__ inline uint32_t QmmMin(uint32_t a, uint32_t b)
{
    return (a < b) ? a : b;
}

// 单个核（组）负责的输出子块
struct QmmCoreTile {
    uint32_t m0;
    uint32_t n0;
    uint32_t curM;
    uint32_t curN;
    bool valid;
};

// 核间切分：blockIdx -> (m0, n0) 子块，M 方向优先连续编号
__aicore__ inline QmmCoreTile QmmCalcTile(uint32_t blockIdx, uint32_t m, uint32_t n, uint32_t singleCoreM,
    uint32_t singleCoreN)
{
    QmmCoreTile tile {0, 0, 0, 0, false};
    uint32_t mBlocks = (m + singleCoreM - 1) / singleCoreM;
    if (mBlocks == 0) {
        return tile;
    }
    uint32_t mIdx = blockIdx % mBlocks;
    uint32_t nIdx = blockIdx / mBlocks;
    tile.m0 = mIdx * singleCoreM;
    tile.n0 = nIdx * singleCoreN;
    if (tile.n0 >= n) {
        return tile;  // tiling 多给的核，无事可做
    }
    tile.curM = QmmMin(singleCoreM, m - tile.m0);
    tile.curN = QmmMin(singleCoreN, n - tile.n0);
    tile.valid = true;
    return tile;
}

inline uint32_t QmmAlignUpHost(uint32_t v, uint32_t a)
{
    return ((v + a - 1) / a) * a;
}

#pragma pack(push, 8)
struct alignas(8) QmmCustomTilingData {
  // 任务一设计说明：
  // cubeTilingData —— CANN Matmul 库标准 tiling（baseM/baseN/baseK、singleCore*、usedCoreNum 等），
  //                  由 host 侧 MultiCoreMatmulTiling 求出后按 baseM/baseN 对齐回填；
  // isPertoken     —— 路径选择：0 = 纯 Cube 输出 INT32；1 = Cube+Vector 反量化输出 BF16；
  // workspaceSize  —— Path2 中 Cube 写 INT32 中间结果所需的 GM 中转区字节数（Path1 仅需非空占位）；
  // m/n/k          —— 全局矩阵形状，kernel 内做越界裁剪与 C 的行跨距计算；
  // singleCoreM/N  —— 核（组）间切分粒度，已在 host 侧按 baseM / n0 对齐，kernel 直接换算子块与偏移。
  TCubeTiling cubeTilingData;
  uint32_t isPertoken;
  uint32_t workspaceSize;
  uint32_t m;
  uint32_t n;
  uint32_t k;
  uint32_t singleCoreM;
  uint32_t singleCoreN;
};
#pragma pack(pop)

// 计算QmmCustom tiling参数,返回tilingData结构体
static QmmCustomTilingData CalcQmmTiling(uint32_t isPertoken, uint32_t M,
                                         uint32_t N, uint32_t K) {
  QmmCustomTilingData tilingData {};
  auto ascendcPlatform =
      platform_ascendc::PlatformAscendCManager::GetInstance();
  // 任务一实现说明：
  // 1) 用 Matmul 库的 MultiCoreMatmulTiling 求 A8W8（A: ND int8 / B: NZ int8 / C: ND int32）的
  //    Cube 分块参数，失败时退回固定 128x128 切分兜底；
  // 2) 库给出的核间切分可能不满足 n0(=baseN) 对齐（如 N=6144 / 20 核 -> singleCoreN=615），
  //    而 B 的 NZ 列块偏移与 C 的写地址都要求按 baseN 对齐，故按 baseM/baseN 重算对齐切分并回填；
  // 3) Path2 需要 M*N*4 字节的 GM 中转区放 Cube 的 INT32 结果；
  // 4) LLM 推理中同一 (M,N,K,路径) 会被高频重复调用，tiling 计算结果按 key 缓存，避免每次
  //    launch 都重跑 tiling 求解（decode 阶段 host 开销会直接变成端到端时延）。
  static std::map<std::array<uint32_t, 4>, QmmCustomTilingData> tilingCache;
  const std::array<uint32_t, 4> key {isPertoken, M, N, K};
  auto cacheIt = tilingCache.find(key);
  if (cacheIt != tilingCache.end()) {
    return cacheIt->second;
  }

  uint32_t aicNum = (ascendcPlatform != nullptr) ? ascendcPlatform->GetCoreNumAic() : 0U;
  if (ascendcPlatform == nullptr || aicNum == 0U) {
    tilingData.workspaceSize = 0;  // kernel 侧的 workspace 空检查会直接安全退出
    return tilingData;
  }
  aicNum = (aicNum == 0U) ? 1U : aicNum;

  matmul_tiling::MultiCoreMatmulTiling mmTiling(*ascendcPlatform);
  mmTiling.SetDim(aicNum);
  mmTiling.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_INT8);
  mmTiling.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::NZ, matmul_tiling::DataType::DT_INT8);
  mmTiling.SetCType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_INT32);
  mmTiling.SetBias(false);
  mmTiling.SetShape(M, N, K);
  mmTiling.SetOrgShape(M, N, K);
  int64_t tilingRet = mmTiling.GetTiling(tilingData.cubeTilingData);
  if (tilingRet != 0) {
    matmul_tiling::MultiCoreMatmulTiling mmTilingFix(*ascendcPlatform);
    mmTilingFix.SetDim(aicNum);
    mmTilingFix.SetAType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_INT8);
    mmTilingFix.SetBType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::NZ, matmul_tiling::DataType::DT_INT8);
    mmTilingFix.SetCType(matmul_tiling::TPosition::GM, matmul_tiling::CubeFormat::ND, matmul_tiling::DataType::DT_INT32);
    mmTilingFix.SetBias(false);
    mmTilingFix.SetShape(M, N, K);
    mmTilingFix.SetOrgShape(M, N, K);
    mmTilingFix.SetFixSplit(128, 128, -1);
    if (mmTilingFix.GetTiling(tilingData.cubeTilingData) != 0) {
      tilingData.isPertoken = isPertoken;
      tilingData.workspaceSize = 0;  // 求解失败：交给 kernel 侧的 workspace 空检查安全退出
      return tilingData;
    }
  }

  // 按 baseM / baseN 对齐重算核间切分，保证 NZ 列偏移与 C 写地址对齐
  uint32_t baseM = (tilingData.cubeTilingData.baseM > 0) ? static_cast<uint32_t>(tilingData.cubeTilingData.baseM) : 16U;
  uint32_t baseN = (tilingData.cubeTilingData.baseN > 0) ? static_cast<uint32_t>(tilingData.cubeTilingData.baseN) : 256U;
  uint32_t mBlocks = (M + baseM - 1) / baseM;
  if (mBlocks > aicNum) {
    mBlocks = aicNum;
  }
  uint32_t nBudget = aicNum / mBlocks;
  if (nBudget == 0) {
    nBudget = 1;
  }
  uint32_t nBlocks = (N + baseN - 1) / baseN;
  if (nBlocks > nBudget) {
    nBlocks = nBudget;
  }
  uint32_t singleCoreM = QmmAlignUpHost((M + mBlocks - 1) / mBlocks, baseM);
  uint32_t singleCoreN = QmmAlignUpHost((N + nBlocks - 1) / nBlocks, baseN);
  // 对齐放大后实际块数可能变少，按最终粒度重算
  mBlocks = (M + singleCoreM - 1) / singleCoreM;
  nBlocks = (N + singleCoreN - 1) / singleCoreN;

  tilingData.cubeTilingData.singleCoreM = static_cast<int32_t>(singleCoreM);
  tilingData.cubeTilingData.singleCoreN = static_cast<int32_t>(singleCoreN);
  tilingData.cubeTilingData.usedCoreNum = static_cast<int32_t>(mBlocks * nBlocks);

  tilingData.isPertoken = isPertoken;
  tilingData.m = M;
  tilingData.n = N;
  tilingData.k = K;
  tilingData.singleCoreM = singleCoreM;
  tilingData.singleCoreN = singleCoreN;
  // Path2: Cube 的 INT32 中间结果按 [M, N] 行主序放 GM 中转区；Path1 只需一个非空占位区
  tilingData.workspaceSize =
      (isPertoken != 0U) ? static_cast<uint32_t>(static_cast<uint64_t>(M) * N * sizeof(int32_t)) : 512U;

  tilingCache[key] = tilingData;
  return tilingData;
}

// 任务二: 实现A8W8输入INT32输出的kernel
// 纯 Cube 路径：每个核（组）负责一个 [singleCoreM, singleCoreN] 子块，
// INT8 x INT8 -> INT32 精确累加后由 fixpipe 直接写回 y（ND, 行跨距 N）。
class QmmCubeBasicKernel {
public:
    __aicore__ inline QmmCubeBasicKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, AscendC::TPipe *tPipe);

    __aicore__ inline void Process();

private:
    QmmMatmul mm;
    GlobalTensor<int8_t> aGm;
    GlobalTensor<int8_t> bGm;
    GlobalTensor<int32_t> cGm;
    const QmmCustomTilingData *td {nullptr};
    QmmCoreTile tile {0, 0, 0, 0, false};
};

__aicore__ inline void QmmCubeBasicKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR y,
    GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData, AscendC::TPipe *tPipe)
{
    // kernel 初始化：Matmul 对象绑定 tiling/TPipe，绑定 A/B/C 的 GM 地址并算出本核子块
    if ASCEND_IS_AIC {
        td = tilingData;
        mm.SetSubBlockIdx(0);
        mm.Init(&tilingData->cubeTilingData, tPipe);
        aGm.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1));
        bGm.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2));
        cGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(y));
        tile = QmmCalcTile(GetBlockIdx(), tilingData->m, tilingData->n, tilingData->singleCoreM,
                           tilingData->singleCoreN);
    }
}

__aicore__ inline void QmmCubeBasicKernel::Process()
{
    // kernel 执行：设置本核子块形状与 A/B/C 偏移后一次 IterateAll 算完整个子块
    if ASCEND_IS_AIC {
        if (!tile.valid) {
            mm.End();
            return;
        }
        // orgShape 用全局 M/N/K：C 按 [M, N] 行主序写出（行跨距 N），A 的行跨距为 K
        mm.SetOrgShape(td->m, td->n, td->k);
        mm.SetSingleShape(tile.curM, tile.curN, td->k);
        mm.SetTensorA(aGm[static_cast<uint64_t>(tile.m0) * td->k]);
        // x2 为 NZ 物理排布（n1 最外层）：列块偏移 = (n0 / n0_unit) * K * n0_unit
        mm.SetTensorB(bGm[static_cast<uint64_t>(tile.n0 / QMM_NZ_N0) * td->k * QMM_NZ_N0]);
        mm.IterateAll(cGm[static_cast<uint64_t>(tile.m0) * td->n + tile.n0]);
        mm.End();
    }
}

// 任务三: 实现A8W8输入, 经过perChannelScale和perTokenScale反量化输出BF16的kernel
// Cube-Vector 混核（1 AIC : 2 AIV）：AIC 把 INT32 结果写 GM 中转区，AIV 读出后
// y = bf16(fp32(C) * perChannelScale[n] * perTokenScale[m])；两侧用跨核 flag 配流水。
class QmmPertokenKernel {
public:
    __aicore__ inline QmmPertokenKernel() {}

    __aicore__ inline void Init(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
                                GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData,
                                AscendC::TPipe *tPipe);

    __aicore__ inline void Process();

private:
    QmmMatmul mm;
    GlobalTensor<int8_t> aGm;
    GlobalTensor<int8_t> bGm;
    GlobalTensor<int32_t> cGm;
    GlobalTensor<float> scaleGm;
    GlobalTensor<float> ptGm;
    GlobalTensor<bfloat16_t> yGm;
    TQue<QuePosition::VECIN, 2> queC;
    TQue<QuePosition::VECOUT, 2> queOut;
    TBuf<TPosition::VECIN> scaleBuf;
    TBuf<TPosition::VECIN> ptBuf;
    TBuf<TPosition::VECCALC> midBuf;
    TBuf<TPosition::VECCALC> tmpBuf;
    const QmmCustomTilingData *td {nullptr};
    QmmCoreTile tile {0, 0, 0, 0, false};
    uint32_t stripLenMax {0};
    uint32_t vecBaseM {0};
};

__aicore__ inline void QmmPertokenKernel::Init(
    GM_ADDR x1, GM_ADDR x2, GM_ADDR scale, GM_ADDR pertokenScale, GM_ADDR y,
    GM_ADDR workSpace, const QmmCustomTilingData *__restrict tilingData,
    AscendC::TPipe *tPipe)
{
    td = tilingData;
    if ASCEND_IS_AIC {
        // Cube 侧：INT32 结果写 GM 中转区 workSpace（不直接写 y）
        mm.SetSubBlockIdx(0);
        mm.Init(&tilingData->cubeTilingData, tPipe);
        aGm.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x1));
        bGm.SetGlobalBuffer(reinterpret_cast<__gm__ int8_t *>(x2));
        cGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(workSpace));
        tile = QmmCalcTile(GetBlockIdx(), tilingData->m, tilingData->n, tilingData->singleCoreM,
                           tilingData->singleCoreN);
    } else if ASCEND_IS_AIV {
        // Vector 侧：本组两个 AIV 共同负责同一个 [curM, curN] 子块，按行块交错分工
        tile = QmmCalcTile(static_cast<uint32_t>(GetBlockIdx() / GetTaskRatio()), tilingData->m, tilingData->n,
                           tilingData->singleCoreM, tilingData->singleCoreN);
        scaleGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(scale));
        ptGm.SetGlobalBuffer(reinterpret_cast<__gm__ float *>(pertokenScale));
        yGm.SetGlobalBuffer(reinterpret_cast<__gm__ bfloat16_t *>(y));
        cGm.SetGlobalBuffer(reinterpret_cast<__gm__ int32_t *>(workSpace));
        if (!tile.valid) {
            return;
        }
        // N 方向按条带处理，UB 按最大条带一次性分配（循环内重复 InitBuffer 会重复占用 UB）
        stripLenMax = QmmMin(QMM_VEC_STRIP_N, tile.curN);
        vecBaseM = QMM_VEC_UB_BUDGET_ELEMS / stripLenMax;
        if (vecBaseM == 0) {
            vecBaseM = 1;
        }
        vecBaseM = QmmMin(vecBaseM, tile.curM);
        tPipe->InitBuffer(queC, 2, vecBaseM * stripLenMax * sizeof(int32_t));
        tPipe->InitBuffer(queOut, 2, vecBaseM * stripLenMax * sizeof(bfloat16_t));
        tPipe->InitBuffer(scaleBuf, stripLenMax * sizeof(float));
        tPipe->InitBuffer(ptBuf, vecBaseM * sizeof(float) + 64U);
        tPipe->InitBuffer(midBuf, vecBaseM * stripLenMax * sizeof(float));
        // AscendDequant 的 sharedTmpBuffer 头部要放 deqScale（align8(calCount) 个 float），
        // 剩余部分作为 src 暂存区且必须 >0（库内断言），故至少 (vecBaseM+1) 行 + 256B 余量
        tPipe->InitBuffer(tmpBuf, (vecBaseM + 1) * stripLenMax * sizeof(float) + 256U);
    }
}

__aicore__ inline void QmmPertokenKernel::Process()
{
    if ASCEND_IS_AIC {
        // Cube：整块 INT32 结果写中转区，经 fixpipe 落 GM 后通知本组 AIV，再等 AIV 读完
        if (!tile.valid) {
            mm.End();
            CrossCoreSetFlag<QMM_SYNC_MODE, PIPE_FIX>(QMM_FLAG_AIC2AIV);
            CrossCoreWaitFlag(QMM_FLAG_AIV2AIC);
            return;
        }
        mm.SetOrgShape(td->m, td->n, td->k);
        mm.SetSingleShape(tile.curM, tile.curN, td->k);
        mm.SetTensorA(aGm[static_cast<uint64_t>(tile.m0) * td->k]);
        mm.SetTensorB(bGm[static_cast<uint64_t>(tile.n0 / QMM_NZ_N0) * td->k * QMM_NZ_N0]);
        mm.IterateAll(cGm[static_cast<uint64_t>(tile.m0) * td->n + tile.n0]);
        mm.End();
        CrossCoreSetFlag<QMM_SYNC_MODE, PIPE_FIX>(QMM_FLAG_AIC2AIV);
        CrossCoreWaitFlag(QMM_FLAG_AIV2AIC);
    } else if ASCEND_IS_AIV {
        CrossCoreWaitFlag(QMM_FLAG_AIC2AIV);
        if (tile.valid) {
            const int64_t subIdx = GetSubBlockIdx();
            const uint32_t n = td->n;
            for (uint32_t stripN0 = 0; stripN0 < tile.curN; stripN0 += QMM_VEC_STRIP_N) {
                uint32_t stripLen = QmmMin(QMM_VEC_STRIP_N, tile.curN - stripN0);
                LocalTensor<float> scaleLocal = scaleBuf.Get<float>(stripLenMax);
                // 当前条带的 per-channel scale 一次性搬入 UB
                DataCopyExtParams scaleCopy {1, static_cast<uint32_t>(stripLen * sizeof(float)), 0, 0, 0};
                DataCopyPadExtParams<float> scalePad {};
                DataCopyPad(scaleLocal, scaleGm[tile.n0 + stripN0], scaleCopy, scalePad);
                PipeBarrier<PIPE_MTE2>();

                uint32_t chunk = 0;
                for (uint32_t r0 = 0; r0 < tile.curM; r0 += vecBaseM, ++chunk) {
                    // 行块在本组 2 个 AIV 间交错分工
                    if (static_cast<int64_t>(chunk % GetTaskRatio()) != subIdx) {
                        continue;
                    }
                    uint32_t rows = QmmMin(vecBaseM, tile.curM - r0);
                    uint64_t gmRow = static_cast<uint64_t>(tile.m0 + r0);

                    LocalTensor<int32_t> cLocal = queC.AllocTensor<int32_t>();
                    DataCopyExtParams cCopy {
                        static_cast<uint16_t>(rows), static_cast<uint32_t>(stripLen * sizeof(int32_t)),
                        static_cast<uint32_t>((n - stripLen) * sizeof(int32_t)), 0, 0};
                    DataCopyPadExtParams<int32_t> cPad {};
                    DataCopyPad(cLocal, cGm[gmRow * n + tile.n0 + stripN0], cCopy, cPad);
                    queC.EnQue(cLocal);

                    LocalTensor<float> ptLocal = ptBuf.Get<float>(vecBaseM);
                    DataCopyExtParams ptCopy {1, static_cast<uint32_t>(rows * sizeof(float)), 0, 0, 0};
                    DataCopyPadExtParams<float> ptPad {};
                    DataCopyPad(ptLocal, ptGm[gmRow], ptCopy, ptPad);

                    LocalTensor<int32_t> cIn = queC.DeQue<int32_t>();
                    LocalTensor<float> mid = midBuf.Get<float>(vecBaseM * stripLen);
                    LocalTensor<uint8_t> tmp =
                        tmpBuf.Get<uint8_t>((vecBaseM + 1) * stripLen * sizeof(float) + 256U);
                    PipeBarrier<PIPE_MTE2>();
                    // per-channel 反量化：mid = fp32(C) * perChannelScale[n]
                    AscendDequant(mid, cIn, scaleLocal, tmp, DequantParams {rows, stripLen, stripLen});
                    // per-token 反量化：mid[r] *= pertokenScale[m0 + r0 + r]
                    for (uint32_t r = 0; r < rows; ++r) {
                        float ptVal = ptLocal.GetValue(r);
                        Muls(mid[r * stripLen], mid[r * stripLen], ptVal, stripLen);
                    }
                    PipeBarrier<PIPE_V>();

                    LocalTensor<bfloat16_t> outLocal = queOut.AllocTensor<bfloat16_t>();
                    Cast(outLocal, mid, RoundMode::CAST_RINT, rows * stripLen);
                    queOut.EnQue(outLocal);
                    LocalTensor<bfloat16_t> out = queOut.DeQue<bfloat16_t>();
                    DataCopyExtParams yCopy {
                        static_cast<uint16_t>(rows), static_cast<uint32_t>(stripLen * sizeof(bfloat16_t)), 0,
                        static_cast<uint32_t>((n - stripLen) * sizeof(bfloat16_t)), 0};
                    DataCopyPad(yGm[gmRow * n + tile.n0 + stripN0], out, yCopy);
                    queOut.FreeTensor(out);
                    queC.FreeTensor(cIn);
                }
            }
        }
        // 通知 Cube 侧：本组中转区已读完，可以复用/释放
        CrossCoreSetFlag<QMM_SYNC_MODE, PIPE_MTE2>(QMM_FLAG_AIV2AIC);
    }
}


__global__ __mix__(1, 2) __aicore__
    void qmm_custom_kernel(GM_ADDR x1, GM_ADDR x2, GM_ADDR scale,
                           GM_ADDR pertoken_scale, GM_ADDR y, GM_ADDR workspace,
                           QmmCustomTilingData tilingData) {
  if (workspace == nullptr) {
    return;
  }
  AscendC::TPipe pipe;

  if (tilingData.isPertoken == 0) {
    QmmCubeBasicKernel kernel;
    kernel.Init(x1, x2, scale, y, workspace, &tilingData, &pipe);
    kernel.Process();
    pipe.Destroy();
  } else {
    QmmPertokenKernel kernel;
    kernel.Init(x1, x2, scale, pertoken_scale, y, workspace, &tilingData,
                &pipe);
    kernel.Process();
    pipe.Destroy();
  }
}

#ifndef ACL_PROF_TENSOR_INFO_DEFINED
#define ACL_PROF_TENSOR_INFO_DEFINED
struct aclprofTensor {
  uint32_t type;     // Tensor类型: 0表示输入, 1表示输出。
  uint32_t format;   // Format类型, 对应aclFormat。
  uint32_t dataType; // 数据类型, 对应aclDataType。
  uint32_t shapeDim; // Shape维度数量, 最大支持8维。
  uint32_t shape[8]; // Shape信息。
};

struct aclprofTensorInfo {
  uint64_t opNameId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint64_t opTypeId; // 通过uint64_t aclprofStr2Id(const char *message)转化
  uint32_t resv;
  uint32_t tensorNum;
  uint32_t kernelType;
  uint32_t blockNums;
  void *stream;
  aclprofTensor *tensors;
};

struct aclprofEventAttributes {
  uint16_t version;
  uint16_t size;
  uint32_t messageType;
  union message {
    aclprofTensorInfo *tensorInfo;
  } message;
};
#endif // ACL_PROF_TENSOR_INFO_DEFINED

#define INPUT(x)                                                               \
  aclprofTensor {                                                              \
    (uint32_t)0,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(x.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(x.scalar_type())),   \
        static_cast<uint32_t>(x.sizes().size()), {                             \
      static_cast<uint32_t>(x.sizes().size() > 0 ? x.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(x.sizes().size() > 1 ? x.sizes().data()[1]     \
                                                     : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 2 ? x.sizes().data()[2]     \
                                                     : 0),                     \
          static_cast<uint32_t>(x.sizes().size() > 3 ? x.sizes().data()[3]     \
                                                     : 0)                      \
    }                                                                          \
  }

#define OUTPUT(z)                                                              \
  aclprofTensor {                                                              \
    (uint32_t)1,                                                               \
        static_cast<uint32_t>(                                                 \
            ascendc_ops::GetFormat(static_cast<int64_t>(z.sizes().size()))),   \
        static_cast<uint32_t>(ascendc_ops::GetAclDataType(z.scalar_type())),   \
        static_cast<uint32_t>(z.sizes().size()), {                             \
      static_cast<uint32_t>(z.sizes().size() > 0 ? z.sizes().data()[0] : 0),   \
          static_cast<uint32_t>(z.sizes().size() > 1 ? z.sizes().data()[1]     \
                                                     : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 2 ? z.sizes().data()[2]     \
                                                     : 0),                     \
          static_cast<uint32_t>(z.sizes().size() > 3 ? z.sizes().data()[3]     \
                                                     : 0)                      \
    }                                                                          \
  }

#define INIT_ACL_PROF_TENSOR_INFO(opName, opType, blockdim, kernelType,        \
                                  tensorInfo, stream, ...)                     \
  uint64_t opNameId = aclprofStr2Id(opName);                                   \
  uint64_t opTypeId = aclprofStr2Id(opType);                                   \
  struct aclprofTensor tensors[] = {__VA_ARGS__};                              \
  tensorInfo = {                                                               \
      opNameId,   opTypeId, 0,      sizeof(tensors) / sizeof(aclprofTensor),   \
      kernelType, blockdim, stream, tensors}

namespace ascendc_ops {

aclDataType GetAclDataType(at::ScalarType scalarType) {
  switch (scalarType) {
  case at::ScalarType::Float:
    return ACL_FLOAT;
  case at::ScalarType::Int:
    return ACL_INT32;
  case at::ScalarType::BFloat16:
    return ACL_BF16;
  case at::ScalarType::Byte:
    return ACL_UINT8;
  case at::ScalarType::Char:
    return ACL_INT8;
  default:
    TORCH_CHECK(false,
                "Unsupported tensor dtype for profiling report: ", scalarType);
  }
}

aclFormat GetFormat(int64_t dimNum) {
  aclFormat format = ACL_FORMAT_ND;
  if (dimNum == 4) {
    format = ACL_FORMAT_FRACTAL_NZ;
  }
  return format;
}

at::Tensor qmm_custom(const at::Tensor &x1, const at::Tensor &x2,
                      const at::Tensor &scale,
                      const c10::optional<at::Tensor> &pertoken_scale) {
  auto aclStream = c10_npu::getCurrentNPUStream().stream(true);

  auto M = x1.size(0);
  auto K = x1.size(1);
  auto N = x2.size(1);

  uint32_t isPertoken =
      pertoken_scale.has_value() && pertoken_scale.value().defined() ? 1 : 0;
  at::ScalarType output_dtype =
      isPertoken ? at::ScalarType::BFloat16 : at::ScalarType::Int;
  at::Tensor y = at::empty({M, N}, x1.options().dtype(output_dtype));

  QmmCustomTilingData tilingData = CalcQmmTiling(isPertoken, M, N, K);
  uint32_t numBlocks = tilingData.cubeTilingData.usedCoreNum;

  at::Tensor workspaceTensor =
      at::empty({static_cast<int64_t>(tilingData.workspaceSize)},
                x1.options().dtype(at::ScalarType::Byte));

  aclprofTensorInfo tensorInfo;
  uint64_t opNameId = aclprofStr2Id("qmm_custom");
  uint64_t opTypeId = aclprofStr2Id("QmmCustom");
  uint8_t *pertokenPtr =
      isPertoken ? (uint8_t *)(pertoken_scale.value().mutable_data_ptr())
                 : nullptr;
  std::vector<aclprofTensor> tensors = {INPUT(x1), INPUT(x2), INPUT(scale)};
  if (isPertoken) {
    tensors.push_back(INPUT(pertoken_scale.value()));
  }
  tensors.push_back(OUTPUT(y));
  tensorInfo = {opNameId, opTypeId,  0,         (uint32_t)tensors.size(),
                0,        numBlocks, aclStream, tensors.data()};

  aclprofEventAttributes attrs = {1, sizeof(aclprofEventAttributes::message), 0,
                                  &tensorInfo};

  aclprofRangePushEx(&attrs);
  qmm_custom_kernel<<<numBlocks, 0, aclStream>>>(
      (uint8_t *)(x1.mutable_data_ptr()), (uint8_t *)(x2.mutable_data_ptr()),
      (uint8_t *)(scale.mutable_data_ptr()), pertokenPtr,
      (uint8_t *)(y.mutable_data_ptr()), workspaceTensor.data_ptr<uint8_t>(),
      tilingData);
  aclprofRangePop();
  return y;
}
} // namespace ascendc_ops

TORCH_LIBRARY(ascendc_ops, m) {
  m.def("qmm_custom(Tensor x1, Tensor x2, Tensor scale, Tensor? "
        "pertoken_scale=None) -> Tensor");
}

TORCH_LIBRARY_IMPL(ascendc_ops, PrivateUse1, m) {
  m.impl("qmm_custom", TORCH_FN(ascendc_ops::qmm_custom));
}


Overwriting /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/qmm_custom.asc


---
## 4 编译自定义算子

### 任务四：成功编译自定义算子


在算子目录下执行编译命令：

In [3]:
!cd {QMM_CUSTOM_DIR} && mkdir -p build && cd build && cmake -DCMAKE_ASC_ARCHITECTURES=dav-2201 .. && make -j;

-- System processer: aarch64
-- CMAKE_ASC_COMPILER: /home/developer/Ascend/cann-9.0.0/bin/bisheng
-- ASCEND_CANN_PACKAGE_LINUX_PATH: /home/developer/Ascend/cann-9.0.0/aarch64-linux
-- CMAKE_ASC_LLD_LINKER: /home/developer/Ascend/cann-9.0.0/aarch64-linux/ccec_compiler/bin/ld.lld


-- The CXX compiler identification is GNU 9.4.0
-- Detecting CXX compiler ABI info


-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done


-- Found Python3: /opt/buildtools/Python-3.11.4/bin/python3 (found version "3.11.4") found components: Interpreter Development Development.Module Development.Embed 


CMake Warning at /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:22 (message):
  static library kineto_LIBRARY-NOTFOUND not found.
Call Stack (most recent call first):
  /home/developer/.local/lib/python3.11/site-packages/torch/share/cmake/Torch/TorchConfig.cmake:125 (append_torchlib_if_found)
  CMakeLists.txt:14 (find_package)


-- Found Torch: /home/developer/.local/lib/python3.11/site-packages/torch/lib/libtorch.so  


-- Configuring done
-- Generating done
-- Build files have been written to: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/qmm_custom/build


[ 50%] Building ASC object CMakeFiles/ascendc_ops.dir/qmm_custom.asc.o


[100%] Linking ASC shared library libascendc_ops.so


[100%] Built target ascendc_ops


编译成功后会在build目录下生成ascendc_ops.so

---
## 5 测试自定义算子功能性能

### 任务五：根据提供的测试脚本，测试自定义算子功能与性能，尝试对自定义算子进行优化后重复整个流程

使用测试脚本验证 QmmCustom 算子在不同规格输入下的精度和性能。测试脚本覆盖多组 (M, K, N) 规格，分别验证 INT32 输出（无 perTokenScale）和 BF16 输出（有 perTokenScale）两种路径。

核心接口调用如下：

```python
output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale_npu)
```

### 5.1 测试算子功能

需要测试的 cases 的 shape 如下，对应的是 Qwen3-8B 模型中实际出现的调用，**全部通过**视为算子功能OK：
| M    | K     | N     |
| ---- | ----- | ----- |
| 1    | 4096  | 4096  |
| 1    | 4096  | 6144  |
| 1    | 4096  | 24576 |
| 1    | 12288 | 4096  |
| 50   | 4096  | 4096  |
| 50   | 4096  | 6144  |
| 50   | 4096  | 24576 |
| 50   | 12288 | 4096  |
| 4096 | 4096  | 4096  |
| 4096 | 4096  | 6144  |
| 4096 | 4096  | 24576 |
| 4096 | 12288 | 4096  |

In [4]:
import torch
import torch_npu
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

torch.npu.config.allow_internal_format = True
torch.npu.config.allow_hf32 = False
torch.npu.set_compile_mode(jit_compile=False)

# ============================================================
# 测试配置: 多组 (M, K, N) 规格
# ============================================================
TEST_SHAPES = [
    (1, 4096, 4096),
    (1, 4096, 6144),
    (1, 4096, 24576),
    (1, 12288, 4096),
    (50, 4096, 4096),
    (50, 4096, 6144),
    (50, 4096, 24576),
    (50, 12288, 4096),
    (4096, 4096, 4096),
    (4096, 4096, 6144),
    (4096, 4096, 24576),
    (4096, 12288, 4096),
]


def check_close(output, ref, dtype):
    """统一的精度比对函数，根据输出 dtype 使用不同的 tolerance"""
    if dtype == torch.int32:
        # 整数运算应精确匹配，tolerance 为 0
        rtol, atol = 0, 0
    else:
        # BF16 允许浮点舍入误差
        rtol, atol = 0.01, 0.01

    output_f = output.to(torch.float32)
    ref_f = ref.to(torch.float32)
    passed = torch.allclose(output_f, ref_f, rtol=rtol, atol=atol)

    label = 'INT32' if dtype == torch.int32 else 'BF16'
    print(f"  {label} allclose验证: {'PASS' if passed else 'FAIL'}")

    if not passed:
        print(f"  output[0,0:5] = {output[0, 0:5].tolist()}")
        print(f"  ref[0,0:5]    = {ref[0, 0:5].tolist()}")

    return passed


def test_int32(M, K, N):
    """无 pertoken_scale, 纯INT8 matmul, INT32 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu, pertoken_scale=None)
    output_cpu = output_npu.cpu()

    # float64 matmul 走 BLAS 优化, 比 int32 朴素乘加快很多;
    # float64 精度足够精确表示 int32 范围内的整数结果
    ref_output = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)

    assert output_cpu.shape == ref_output.shape, f"shape不一致: {output_cpu.shape} vs {ref_output.shape}"
    assert output_cpu.dtype == torch.int32, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_output, torch.int32)


def test_bf16(M, K, N):
    """有 pertoken_scale, INT8 matmul + dequant, BF16 输出"""
    x1_cpu = torch.randint(-128, 127, (M, K), dtype=torch.int8)
    x2_cpu = torch.randint(-128, 127, (K, N), dtype=torch.int8)
    scale_cpu = torch.randn([N], dtype=torch.float32)
    pertoken_scale_cpu = torch.randn([M], dtype=torch.float32)

    x1_npu = x1_cpu.npu()
    x2_npu = x2_cpu.npu()
    x2_npu = torch_npu.npu_format_cast(x2_npu, 29)  # 29: FORMAT_FRACTAL_NZ
    scale_npu = scale_cpu.npu()
    pertoken_scale_npu = pertoken_scale_cpu.npu()

    output_npu = torch.ops.ascendc_ops.qmm_custom(x1_npu, x2_npu, scale_npu,
                                                  pertoken_scale=pertoken_scale_npu)
    output_cpu = output_npu.cpu()

    int32_result = torch.matmul(x1_cpu.to(torch.float64), x2_cpu.to(torch.float64)).round().to(torch.int32)
    ref_bf16 = (int32_result.to(torch.float32) * scale_cpu.unsqueeze(0) * pertoken_scale_cpu.unsqueeze(1)).to(torch.bfloat16)

    assert output_cpu.shape == ref_bf16.shape, f"shape不一致: {output_cpu.shape} vs {ref_bf16.shape}"
    assert output_cpu.dtype == torch.bfloat16, f"输出dtype不正确: {output_cpu.dtype}"

    return check_close(output_cpu, ref_bf16, torch.bfloat16)


# ============================================================
# 主循环: 对每组 (M, K, N) 执行 INT32 和 BF16 测试
# ============================================================
int32_results = []
bf16_results = []

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")
print("=" * 60)


/home/developer/.local/lib/python3.11/site-packages/torch_npu/__init__.py:320: UserWarning: On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default.                      Do not set it to 1 to prevent some unknown errors
  warnings.warn("On the interactive interface, the value of TASK_QUEUE_ENABLE is set to 0 by default. \


规格1: M=1, K=4096, N=4096
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]
  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


...

  BF16 allclose验证: PASS
规格11: M=4096, K=4096, N=24576
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS
规格12: M=4096, K=12288, N=4096
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS

测试汇总
                      规格 |  INT32   |   BF16  
----------------------------------------------
       M=1,K=4096,N=4096 |   PASS   |   PASS  
       M=1,K=4096,N=6144 |   PASS   |   PASS  
      M=1,K=4096,N=24576 |   PASS   |   PASS  
      M=1,K=12288,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=6144 |   PASS   |   PASS  
     M=50,K=4096,N=24576 |   PASS   |   PASS  
     M=50,K=12288,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=6144 |   PASS   |   PASS  
   M=4096,K=4096,N=24576 |   PASS   |   PASS  
   M=4096,K=12288,N=4096 |   PASS   |   PASS  
INT32: 12/12 通过, BF16: 12/12 通过


### 5.2 测试算子性能
基于torch_npu.profiler来测试算子性能，执行下面的code cell测试单算子性能数据：

In [5]:
import os
import csv
from pathlib import Path

# ============================================================
# Profiler 配置
# ============================================================
PROF_OUTPUT_DIR = str(SRC_DIR / "prof_output")  # 采集数据的存放路径
WARMUP_ITERS = 0      # warmup 步数, 不采集数据

def parse_kernel_details(csv_path):
    """解析 kernel_details.csv, 提取 QmmCustom 每次执行的 Duration
    """
    rows = []
    with open(csv_path, newline="", encoding="utf-8", errors="replace") as f:
        reader = csv.DictReader(f)
        for row in reader:
            name = row.get("Name", "")
            if "QmmCustom" not in name:
                continue

            # 解析 Input Shapes: 格式如 "1,4096;4096,4096;4096;"
            raw_shapes = row.get("Input Shapes", "").strip('"')
            parts = [p.strip() for p in raw_shapes.split(";")]
            x1_shape = parts[0].split(",")  # (M, K)
            x2_shape = parts[1].split(",")  # (K, N)
            M = int(x1_shape[0])
            K = int(x1_shape[1])
            N = int(x2_shape[1])

            # 从 Output Data Types 判断 dtype
            out_dtype_raw = row.get("Output Data Types", "").strip('"')
            dtype_tag = "INT32" if "INT32" in out_dtype_raw else "BF16"

            # Duration(us)
            duration_us = float(row.get("Duration(us)", "0") or "0")

            rows.append({
                "M": M, "K": K, "N": N,
                "dtype": dtype_tag,
                "duration_us": duration_us,
            })
    return rows


def find_kernel_details_csv(prof_dir):
    prof_path = Path(prof_dir)
    if not prof_path.exists():
        return None
    # 找出所有包含 ASCEND_PROFILER_OUTPUT/kernel_details.csv 的子目录
    candidates = []
    for subdir in sorted(prof_path.iterdir(), key=lambda d: d.name, reverse=True):
        csv_path = subdir / "ASCEND_PROFILER_OUTPUT" / "kernel_details.csv"
        if csv_path.exists():
            candidates.append(csv_path)
            break  # sorted reverse=True, 第一个就是最新的
    return str(candidates[0]) if candidates else None

# ============================================================
# 主流程: 先 warmup, 再开启 profiler 采集, 最后汇总
# ============================================================
int32_results = []
bf16_results = []

# warmup: 不采集数据, 让 NPU 稳定
if WARMUP_ITERS>0:
    print("=" * 60)
    print(f"Warmup: 运行 {WARMUP_ITERS} 轮不采集数据")
    print("=" * 60)
    for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
        test_int32(M, K, N)
        test_bf16(M, K, N)

# 创建 profiler
os.makedirs(PROF_OUTPUT_DIR, exist_ok=True)
profiler = torch_npu.profiler.profile(
    activities=[
        torch_npu.profiler.ProfilerActivity.NPU,
        torch_npu.profiler.ProfilerActivity.CPU,
    ],
    experimental_config=torch_npu.profiler._ExperimentalConfig(
        profiler_level=torch_npu.profiler.ProfilerLevel.Level1,
        aic_metrics=torch_npu.profiler.AiCMetrics.PipeUtilization,
    ),
    on_trace_ready=torch_npu.profiler.tensorboard_trace_handler(PROF_OUTPUT_DIR),
)

# 开启 profiler 采集
profiler.start()

for i, (M, K, N) in enumerate(TEST_SHAPES, 1):
    print("=" * 60)
    print(f"规格{i}: M={M}, K={K}, N={N}")
    print("=" * 60)

    print("[INT32 测试]")
    int32_pass = test_int32(M, K, N)
    int32_results.append((M, K, N, int32_pass))
    profiler.step()

    print("[BF16 测试]")
    bf16_pass = test_bf16(M, K, N)
    bf16_results.append((M, K, N, bf16_pass))
    profiler.step()

# 停止 profiler
profiler.stop()

# ============================================================
# 测试汇总
# ============================================================
print()
print("=" * 60)
print("测试汇总")
print("=" * 60)
print(f"{'规格':>24} | {'INT32':^8} | {'BF16':^8}")
print("-" * 46)
for (M, K, N, p1), (_, _, _, p2) in zip(int32_results, bf16_results):
    tag = f"M={M},K={K},N={N}"
    print(f"{tag:>24} | {'PASS' if p1 else 'FAIL':^8} | {'PASS' if p2 else 'FAIL':^8}")

int32_total = sum(p for _, _, _, p in int32_results)
bf16_total = sum(p for _, _, _, p in bf16_results)
total = len(TEST_SHAPES)
print(f"INT32: {int32_total}/{total} 通过, BF16: {bf16_total}/{total} 通过")

# ============================================================
# Profiler 数据汇总: 每种规格每种 dtype 的 Duration(us)
# ============================================================
print()
print("=" * 70)
print("Profiler 数据汇总 (QmmCustom Duration)")
print("=" * 70)

csv_path = find_kernel_details_csv(PROF_OUTPUT_DIR)
print(f"原始数据路径：{csv_path}")
if csv_path:
    kernel_rows = parse_kernel_details(csv_path)
    if kernel_rows:
        # 按 (M, K, N) 分组, 每组内有 INT32 和 BF16
        shape_map = {}
        for kr in kernel_rows:
            key = (kr["M"], kr["K"], kr["N"])
            shape_map.setdefault(key, {})[kr["dtype"]] = kr["duration_us"]

        print(f"{'规格 (M,K,N)':>24} | {'INT32 Duration(us)':>18} | {'BF16 Duration(us)':>18}")
        print("-" * 70)
        for M, K, N in TEST_SHAPES:
            key = (M, K, N)
            int32_dur = shape_map.get(key, {}).get("INT32", "-")
            bf16_dur = shape_map.get(key, {}).get("BF16", "-")
            int32_str = f"{int32_dur:.3f}" if isinstance(int32_dur, float) else int32_dur
            bf16_str = f"{bf16_dur:.3f}" if isinstance(bf16_dur, float) else bf16_dur
            tag = f"M={M},K={K},N={N}"
            print(f"{tag:>24} | {int32_str:>18} | {bf16_str:>18}")
    else:
        print(f"在 {csv_path} 中未找到 QmmCustom 算子数据")
else:
    print(f"未在 {PROF_OUTPUT_DIR} 中找到 kernel_details.csv 文件")
    print("请检查 profiler 输出目录")

print("=" * 70)


规格1: M=1, K=4096, N=4096
[INT32 测试]
  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS
规格2: M=1, K=4096, N=6144
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


  BF16 allclose验证: PASS
规格3: M=1, K=4096, N=24576
[INT32 测试]


  INT32 allclose验证: PASS
[BF16 测试]


...

/

-

[2026-09-10 17:05:41] [INFO] [29994] profiler.py: CANN profiling data parsed in a total time of 0:00:02.019008

[2026-09-10 17:05:41] [INFO] [29962] profiler.py: All profiling data parsed in a total time of 0:00:03.031921


测试汇总
                      规格 |  INT32   |   BF16  
----------------------------------------------
       M=1,K=4096,N=4096 |   PASS   |   PASS  
       M=1,K=4096,N=6144 |   PASS   |   PASS  
      M=1,K=4096,N=24576 |   PASS   |   PASS  
      M=1,K=12288,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=4096 |   PASS   |   PASS  
      M=50,K=4096,N=6144 |   PASS   |   PASS  
     M=50,K=4096,N=24576 |   PASS   |   PASS  
     M=50,K=12288,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=4096 |   PASS   |   PASS  
    M=4096,K=4096,N=6144 |   PASS   |   PASS  
   M=4096,K=4096,N=24576 |   PASS   |   PASS  
   M=4096,K=12288,N=4096 |   PASS   |   PASS  
INT32: 12/12 通过, BF16: 12/12 通过

Profiler 数据汇总 (QmmCustom Duration)
原始数据路径：/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/op_custom/prof_output/b7247e50141e44d6b9386800a260f93c_27885_20260910170525701_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
              规格 (M,K,N) | INT32 Duration(us) |

---
## 6 自定义算子接入模型

### 6.1 定位替换点

Qwen3-8B 推理框架中的 W8A8 量化 matmul 通过 `CompressedTensorsW8A8Int8LinearMethod` 类实现，其 `apply_weights` 方法中使用 `torch_npu.npu_quant_matmul` 执行 INT8 量化矩阵乘法。我们需要将这个调用替换为自定义的 `torch.ops.ascendc_ops.qmm_custom` 算子。

查看原始实现：

In [6]:
!grep -n 'npu_quant_matmul' {RECIPE_ROOT}/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py

93:        x = torch_npu.npu_quant_matmul(x, layer.weight,


### 6.2 替换实现

在 `CompressedTensorsW8A8Int8LinearMethod` 中，将 `torch_npu.npu_quant_matmul` 替换为 `torch.ops.ascendc_ops.qmm_custom`。

替换后的方法为：

```python
torch.ops.load_library(str(QMM_CUSTOM_DIR / "build/libascendc_ops.so"))

def apply_weights(self, layer: torch.nn.Module,
                x: torch.Tensor,
                bias: Optional[torch.Tensor],
                dynamic_scale: Optional = None,
                out_dtype: Optional = torch.bfloat16
                ) -> Union[torch.Tensor, Dict[str, Any]]:
    if dynamic_scale is not None:
        x_scale = dynamic_scale
    else:
        x, x_scale = torch_npu.npu_dynamic_quant(x, smooth_scales=layer.smooth_scales)
    out_shape_dim_0 = x.size()[:-1]
    x = x.view(-1, x.size(-1))
    x_scale = x_scale.view(-1)

    # --- 替换核心: npu_quant_matmul → qmm_custom ---
    x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight, 
                                layer.weight_scale.view(-1),
                                pertoken_scale=None if out_dtype == torch.int32 else x_scale)
    # --- 替换结束 ---

    x = x.view(out_shape_dim_0 + (-1, ))
    if out_dtype == torch.int32:
        return x, x_scale
    else:
        return x
```

执行如下操作替换推理网络中调用的接口实现：

In [7]:
# 6.2 替换实现: 将 torch_npu.npu_quant_matmul 替换为 torch.ops.ascendc_ops.qmm_custom

# 1: 导入libascendc_ops.so
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
print('[替换] 目标文件:', W8A8_FILE)

# 备份原始内容，仅在首次执行时保存，避免重复执行时覆盖为已修改的代码
if 'W8A8_ORIGINAL_CODE' not in dir():
    W8A8_ORIGINAL_CODE = W8A8_FILE.read_text(encoding='utf-8')

ct_code = W8A8_FILE.read_text(encoding='utf-8')
_lib_path = str(QMM_CUSTOM_DIR / "build/libascendc_ops.so")
_load_lib_stmt = 'torch.ops.load_library(r"' + _lib_path + '")'
if _load_lib_stmt not in ct_code:
    # 在 'import torch' 行后插入 'torch.ops.load_library'
    ct_code = ct_code.replace('import torch\n', 'import torch\n' + _load_lib_stmt + '\n')
    W8A8_FILE.write_text(ct_code, encoding='utf-8')
    print('[插入] compressed_tensors.py: 已添加 torch.ops.load_library')
else:
    print('[插入] compressed_tensors.py: torch.ops.load_library 已存在，跳过')

# 2: 在 compressed_tensors_w8a8_int8.py 中替换 npu_quant_matmul → qmm_custom
code = W8A8_FILE.read_text(encoding='utf-8')

# 替换 npu_quant_matmul 调用为 qmm_custom 调用
old_call = (
    'x = torch_npu.npu_quant_matmul(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale,\n'
    '                                    bias=layer.bias,\n'
    '                                    output_dtype=out_dtype)'
)
new_call = (
    '# --- 替换核心: npu_quant_matmul → qmm_custom ---\n'
    '        x = torch.ops.ascendc_ops.qmm_custom(x, layer.weight,\n'
    '                                    layer.weight_scale.view(-1),\n'
    '                                    pertoken_scale=None if out_dtype == torch.int32 else x_scale)\n'
    '        # --- 替换结束 ---'
)

if old_call in code:
    code = code.replace(old_call, new_call)
    W8A8_FILE.write_text(code, encoding='utf-8')
    print('[替换] 成功: npu_quant_matmul → qmm_custom')
else:
    print('[替换] 文件中未找到 npu_quant_matmul 调用，可能已替换')

# 验证替换结果
result = W8A8_FILE.read_text(encoding='utf-8')
if 'torch.ops.ascendc_ops.qmm_custom' in result:
    print('[验证] 替换已生效，当前使用 qmm_custom 算子')
else:
    print('[验证] 替换未生效，请检查文件内容')


[替换] 目标文件: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py
[插入] compressed_tensors.py: 已添加 torch.ops.load_library
[替换] 成功: npu_quant_matmul → qmm_custom
[验证] 替换已生效，当前使用 qmm_custom 算子


### 6.3 替换实现后执行量化推理

指定本地的量化模型权重路径，调用infer.sh执行量化推理，查看 W8A8 量化模型推理生成的文本，对比替换前的生成文本是否一致。

**注意：需要先完成第五章的量化模型推理过程，并将`W8A8_WEIGHT`指定为本地的量化模型权重的路径**

In [8]:
import sys, subprocess, os
from inference_scripts.recipe_workflow import prepare_runtime_yaml

# 将W8A8_WEIGHT设置为量化模型权重实际所在路径
W8A8_WEIGHT = TUTORIAL_DIR / 'src/data/models/Qwen3-8B-W8A8'

YAML_A8W8_TEMPLATE =  RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_1tp.yaml'
YAML_A8W8_CUSTOM  = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom.yaml'
prepare_runtime_yaml(YAML_A8W8_TEMPLATE, YAML_A8W8_CUSTOM, model_path=str(W8A8_WEIGHT))

env = os.environ.copy()
python_dir = os.path.dirname(sys.executable)
env["PATH"] = python_dir + ":" + env.get("PATH", "")
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom.yaml)
world_size = 1


[INFO][2026-09-10 17:05:50.527][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-09-10 17:05:50.527][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-09-10 17:05:50.527][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-09-10 17:05:50.530][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-09-10 17:05:50.530][patch_getenv.py:15] get env RES_PATH = res/20260910/qwen3_8b_qwen3_8b_a8w8_custom
[INFO][2026-09-10 17:05:50.530][infer.py:146] Inference Configuration
[INFO][2026-09-10 17:05:50.530][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8', output_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260

[INFO][2026-09-10 17:05:51.458][execution_engine.py:126] Loading main model...
/home/developer/.local/lib/python3.11/site-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[INFO][2026-09-10 17:05:51.547][model_worker.py:185] Try to load pretrained model in path: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8
[INFO][2026-09-10 17:05:51.547][comm_manager.py:442] CommManager: group 'attn_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_group=0)
[INFO][2026-09-10 17:05:51.547][comm_manager.py:442] CommManager: group 'lmhead_tp_group' registered (my

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading safetensors checkpoint shards:   0% Completed | 0/37 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  24% Completed | 9/37 [00:00<00:00, 80.07it/s]


...

[INFO][2026-09-10 17:06:23.783][scheduler.py:518] [Main] Inference time (decode): 37.83 ms
[INFO][2026-09-10 17:06:23.823][scheduler.py:518] [Main] Inference time (decode): 37.73 ms
[INFO][2026-09-10 17:06:23.862][scheduler.py:518] [Main] Inference time (decode): 37.87 ms
[INFO][2026-09-10 17:06:23.901][scheduler.py:518] [Main] Inference time (decode): 37.54 ms
[INFO][2026-09-10 17:06:23.940][scheduler.py:518] [Main] Inference time (decode): 37.54 ms
[INFO][2026-09-10 17:06:23.980][scheduler.py:518] [Main] Inference time (decode): 37.80 ms


[INFO][2026-09-10 17:06:24.019][scheduler.py:518] [Main] Inference time (decode): 37.96 ms
[INFO][2026-09-10 17:06:24.060][scheduler.py:518] [Main] Inference time (decode): 39.52 ms
[INFO][2026-09-10 17:06:24.100][scheduler.py:518] [Main] Inference time (decode): 37.94 ms
[INFO][2026-09-10 17:06:24.139][scheduler.py:518] [Main] Inference time (decode): 37.56 ms
[INFO][2026-09-10 17:06:24.178][scheduler.py:518] [Main] Inference time (decode): 37.54 ms
[INFO][2026-09-10 17:06:24.217][scheduler.py:518] [Main] Inference time (decode): 37.67 ms


[INFO][2026-09-10 17:06:24.256][scheduler.py:518] [Main] Inference time (decode): 37.34 ms
[INFO][2026-09-10 17:06:24.295][scheduler.py:518] [Main] Inference time (decode): 37.65 ms
[INFO][2026-09-10 17:06:24.334][scheduler.py:518] [Main] Inference time (decode): 37.56 ms
[INFO][2026-09-10 17:06:24.373][scheduler.py:518] [Main] Inference time (decode): 37.47 ms
[INFO][2026-09-10 17:06:24.413][scheduler.py:518] [Main] Inference time (decode): 37.70 ms
[INFO][2026-09-10 17:06:24.452][scheduler.py:518] [Main] Inference time (decode): 37.62 ms


[INFO][2026-09-10 17:06:24.491][scheduler.py:518] [Main] Inference time (decode): 37.71 ms
[INFO][2026-09-10 17:06:24.530][scheduler.py:518] [Main] Inference time (decode): 37.50 ms
[INFO][2026-09-10 17:06:24.569][scheduler.py:518] [Main] Inference time (decode): 37.48 ms
[INFO][2026-09-10 17:06:24.608][scheduler.py:518] [Main] Inference time (decode): 37.38 ms
[INFO][2026-09-10 17:06:24.647][scheduler.py:518] [Main] Inference time (decode): 37.62 ms
[INFO][2026-09-10 17:06:24.686][scheduler.py:518] [Main] Inference time (decode): 37.50 ms


[INFO][2026-09-10 17:06:24.725][scheduler.py:518] [Main] Inference time (decode): 37.65 ms
[INFO][2026-09-10 17:06:24.765][scheduler.py:518] [Main] Inference time (decode): 37.76 ms
[INFO][2026-09-10 17:06:24.804][scheduler.py:518] [Main] Inference time (decode): 37.58 ms
[INFO][2026-09-10 17:06:24.843][scheduler.py:518] [Main] Inference time (decode): 37.58 ms
[INFO][2026-09-10 17:06:24.843][infer.py:99] Request 0: outputs: The output of an **attention function** is typically a **weighted sum** of the **value vectors**, where the weights are determined by the **similarity** between the **query vector** and the **key vectors** from the set of key-value pairs.

### Formal Description:

Given:
- A **query vector** $ Q $
- A set of **key-value pairs**: $ (K_1, V_1), (K_2, V_2), \dots, (K_n, V_n) $

The **attention function** computes the output as:

$$
\text{Attention}(Q, K, V) = \sum_{i=1}^{n} \alpha_i V_i
$$

Where:
- $ \alpha_i = \text{softmax}\left( \frac{Q \cdot K_i}{\sqrt{d_k}} \rig

---
## 7 模型性能测试

### 任务六：测试模型性能，尝试对自定义算子进行优化后重复上述流程

将 QmmCustom 算子接入模型后，使用 Profiling 工具观测模型推理性能，对比自定义算子替换前后的 matmul 耗时变化。

### 7.1 准备 Profiling YAML

沿用 recipes 的 YAML + `infer.sh` 工作流，在配置中打开 Profiler，运行一次推理并收集性能数据。

通过如下操作将 YAML 配置中的enable_profiler配置为 True ：

In [9]:
from inference_scripts.recipe_workflow import prepare_profiling_yaml

YAML_A8W8_CUSTOM_PROFILER = RECIPE_ROOT / 'models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml'

prepare_profiling_yaml(YAML_A8W8_CUSTOM, YAML_A8W8_CUSTOM_PROFILER, enable_profiler=True)

{'model_config': {'model_name': 'qwen3_8b',
  'model_path': '/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8',
  'exe_mode': 'eager',
  'enable_profiler': True,
  'with_ckpt': True,
  'enable_cache_compile': False,
  'enable_static_kernel': False},
 'data_config': {'dataset': 'default', 'input_truncated_len': 4096},
 'parallel_config': {'world_size': 1,
  'attn_tp_size': 1,
  'moe_tp_size': 1,
  'embed_tp_size': 1,
  'lmhead_tp_size': 1},
 'scheduler_config': {'max_new_tokens': 256, 'batch_size': 1}}

### 7.2 启动 Profiling 推理

使用 recipes 命令启动带 Profiler 的推理，推理结束后会在结果目录中生成 `prof/` 产物。

In [10]:
result = subprocess.run(
    ['bash', 'executor/scripts/infer.sh', '--model', 'qwen', '--yaml', YAML_A8W8_CUSTOM_PROFILER],
    cwd=str(RECIPE_ROOT),
    env=env,
    check=True,
)

rm: cannot remove '/root/atc_data/': Permission denied


====================> 启动离线推理 (model=qwen, yaml=/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/config/qwen3_8b_a8w8_custom_profiler.yaml)
world_size = 1


[INFO][2026-09-10 17:06:39.725][patch_getenv.py:15] get env LOCAL_RANK = 0
[INFO][2026-09-10 17:06:39.725][patch_getenv.py:15] get env RANK_OFFSET = 0
[INFO][2026-09-10 17:06:39.725][patch_getenv.py:15] get env LC_ALL = en_US.UTF-8
[INFO][2026-09-10 17:06:39.727][patch_getenv.py:15] get env WORK_DIR = /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen
[INFO][2026-09-10 17:06:39.727][patch_getenv.py:15] get env RES_PATH = res/20260910/qwen3_8b_qwen3_8b_a8w8_custom_profiler
[INFO][2026-09-10 17:06:39.727][infer.py:146] Inference Configuration
[INFO][2026-09-10 17:06:39.727][infer.py:147] InferenceConfig(model_config=ModelConfig(model_name='qwen3_8b', model_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8', output_path='/mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/

[INFO][2026-09-10 17:06:40.656][execution_engine.py:126] Loading main model...
/home/developer/.local/lib/python3.11/site-packages/transformers/modeling_rope_utils.py:1036: FutureWarning: `rope_config_validation` is deprecated and has been removed. Its functionality has been moved to RotaryEmbeddingConfigMixin.validate_rope method. PreTrainedConfig inherits this class, so please call self.validate_rope() instead. Also, make sure to use the new rope_parameters syntax. You can call self.standardize_rope_params() in the meantime.
  warnings.warn(
[INFO][2026-09-10 17:06:40.743][model_worker.py:185] Try to load pretrained model in path: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/data/models/Qwen3-8B-W8A8
[INFO][2026-09-10 17:06:40.743][comm_manager.py:442] CommManager: group 'attn_tp_group' registered (my_subgroup=[0], reuse=True, rank_in_group=0)
[INFO][2026-09-10 17:06:40.743][comm_manager.py:442] CommManager: group 'lmhead_tp_group' registered (my

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading safetensors checkpoint shards:   0% Completed | 0/37 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  24% Completed | 9/37 [00:00<00:00, 83.81it/s]


...

[INFO][2026-09-10 17:07:22.775][scheduler.py:518] [Main] Inference time (decode): 36.61 ms
[INFO][2026-09-10 17:07:22.813][scheduler.py:518] [Main] Inference time (decode): 36.74 ms
[INFO][2026-09-10 17:07:22.851][scheduler.py:518] [Main] Inference time (decode): 36.78 ms
[INFO][2026-09-10 17:07:22.890][scheduler.py:518] [Main] Inference time (decode): 36.60 ms
[INFO][2026-09-10 17:07:22.928][scheduler.py:518] [Main] Inference time (decode): 36.59 ms
[INFO][2026-09-10 17:07:22.966][scheduler.py:518] [Main] Inference time (decode): 36.54 ms


[INFO][2026-09-10 17:07:23.004][scheduler.py:518] [Main] Inference time (decode): 36.39 ms
[INFO][2026-09-10 17:07:23.042][scheduler.py:518] [Main] Inference time (decode): 36.35 ms
[INFO][2026-09-10 17:07:23.081][scheduler.py:518] [Main] Inference time (decode): 38.03 ms
[INFO][2026-09-10 17:07:23.135][scheduler.py:518] [Main] Inference time (decode): 51.65 ms
[INFO][2026-09-10 17:07:23.183][scheduler.py:518] [Main] Inference time (decode): 46.47 ms


[INFO][2026-09-10 17:07:23.221][scheduler.py:518] [Main] Inference time (decode): 36.30 ms
[INFO][2026-09-10 17:07:23.259][scheduler.py:518] [Main] Inference time (decode): 36.48 ms
[INFO][2026-09-10 17:07:23.297][scheduler.py:518] [Main] Inference time (decode): 36.51 ms
[INFO][2026-09-10 17:07:23.335][scheduler.py:518] [Main] Inference time (decode): 36.52 ms
[INFO][2026-09-10 17:07:23.373][scheduler.py:518] [Main] Inference time (decode): 36.47 ms
[INFO][2026-09-10 17:07:23.411][scheduler.py:518] [Main] Inference time (decode): 36.40 ms


[INFO][2026-09-10 17:07:23.449][scheduler.py:518] [Main] Inference time (decode): 36.31 ms
[INFO][2026-09-10 17:07:23.487][scheduler.py:518] [Main] Inference time (decode): 36.40 ms
[INFO][2026-09-10 17:07:23.525][scheduler.py:518] [Main] Inference time (decode): 36.23 ms
[INFO][2026-09-10 17:07:23.563][scheduler.py:518] [Main] Inference time (decode): 36.48 ms
[INFO][2026-09-10 17:07:23.600][scheduler.py:518] [Main] Inference time (decode): 36.37 ms
[INFO][2026-09-10 17:07:23.638][scheduler.py:518] [Main] Inference time (decode): 36.46 ms


[INFO][2026-09-10 17:07:23.677][scheduler.py:518] [Main] Inference time (decode): 36.50 ms
[INFO][2026-09-10 17:07:23.677][infer.py:99] Request 0: outputs: The output of an **attention function** is typically a **weighted sum** of the **value vectors**, where the weights are determined by the **similarity** between the **query vector** and the **key vectors** from the set of key-value pairs.

### Formal Description:

Given:
- A **query vector** $ Q $
- A set of **key-value pairs**: $ (K_1, V_1), (K_2, V_2), \dots, (K_n, V_n) $

The **attention function** computes the output as:

$$
\text{Attention}(Q, K, V) = \sum_{i=1}^{n} \alpha_i V_i
$$

Where:
- $ \alpha_i = \text{softmax}\left( \frac{Q \cdot K_i}{\sqrt{d_k}} \right) $
- $ d_k $ is the dimensionality of the key vectors (used for scaling to prevent large values)
- $ \cdot $ denotes the dot product
- $ \text{softmax} $ ensures that the weights $ \alpha_i $ sum to 1 and represent attention distribution

### Final Output:
The **output*

### 7.3 查看 Profiling 结果

推理完成后，定位 `prof/` 目录，重点关注其中的 `op_statistic.csv` 和 `kernel_details.csv` 这两个文件。`op_statistic.csv`中展示了模型调用的所有算子的统计信息，可快速确认自定义算子 QmmCustom 的调用情况。`kernel_details.csv`中展示了详细的接口调用信息，可自行查看筛选自己关注的算子的性能。

通过如下操作统计 `kernel_details.csv` 中的 QmmCustom 算子耗时，可自行对比自定义算子替换前后的性能差异。

In [11]:
import pandas as pd
from pathlib import Path

RES_DIR = RECIPE_ROOT / 'models/qwen/res'

# 取最新的 profiling 结果目录
result_dirs = sorted([d for d in RES_DIR.iterdir() if d.is_dir()], key=lambda d: d.name, reverse=True)
latest_result = result_dirs[0]
print(f'最新结果目录: {latest_result}')

def find_kernel_csv(phase_dir):
    matches = sorted(phase_dir.rglob('kernel_details.csv'), key=lambda p: p.parent.parent.name, reverse=True)
    return matches[0] if matches else None

prefill_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill')
decode_csv = find_kernel_csv(latest_result / 'qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode')
assert prefill_csv and decode_csv, 'kernel_details.csv 未找到，请确认 profiling 已完成'
print(f'Prefill CSV: {prefill_csv}')
print(f'Decode CSV: {decode_csv}')

# 读取 CSV 并筛选 QmmCustom 算子
prefill_df = pd.read_csv(prefill_csv)
decode_df = pd.read_csv(decode_csv)

prefill_qmm = prefill_df[prefill_df['Type'].str.contains('QmmCustom')].copy()
decode_qmm = decode_df[decode_df['Type'].str.contains('QmmCustom')].copy()

# 按 Input Shapes 分组，统计每种 shape 下的平均 Duration
def qmm_duration_stats(df, phase):
    stats = df.groupby(['Input Shapes', 'Output Data Types'])['Duration(us)'].agg(['mean', 'count']).reset_index()
    stats.columns = ['Input Shapes', 'Output Dtype', 'Avg Duration(us)', 'Calls']
    stats = stats.sort_values('Avg Duration(us)', ascending=False).reset_index(drop=True)
    stats.insert(0, 'Phase', phase)
    return stats

prefill_stats = qmm_duration_stats(prefill_qmm, 'Prefill')
decode_stats = qmm_duration_stats(decode_qmm, 'Decode')

combined = pd.concat([prefill_stats, decode_stats], ignore_index=True)
display(combined.style.set_caption('QmmCustom 各 Shape 平均耗时 (us)'))

最新结果目录: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260910
Prefill CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260910/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/prefill/b7247e50141e44d6b9386800a260f93c_31734_20260910170704431_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv
Decode CSV: /mnt/workspace/gitCode/cann/cann-learning-hub/tutorials/llm_inference/qwen3_8b/src/inference_scripts/recipe_qwen3_8b/models/qwen/res/20260910/qwen3_8b_qwen3_8b_a8w8_custom_profiler/prof/decode/b7247e50141e44d6b9386800a260f93c_31734_20260910170709197_ascend_pt/ASCEND_PROFILER_OUTPUT/kernel_details.csv


,Phase,Input Shapes,Output Dtype,Avg Duration(us),Calls
0,Prefill,"""50,4096;4096,24576;24576""",INT32,100.623889,36
1,Prefill,"""50,12288;12288,4096;4096;50""",DT_BF16,60.039444,36
2,Prefill,"""50,4096;4096,6144;6144;50""",DT_BF16,52.996111,36
3,Prefill,"""50,4096;4096,4096;4096;50""",DT_BF16,28.636667,36
4,Decode,"""1,4096;4096,24576;24576""",INT32,82.633148,108
5,Decode,"""1,12288;12288,4096;4096;1""",DT_BF16,51.095556,108
6,Decode,"""1,4096;4096,6144;6144;1""",DT_BF16,31.924444,108
7,Decode,"""1,4096;4096,4096;4096;1""",DT_BF16,25.510741,108


### 7.4 恢复原始 W8A8 源码

自定义算子推理与 Profiling 已完成，下方单元格会从备份恢复 `compressed_tensors_w8a8_int8.py` 原始内容，避免修改后的源码污染 baseline 对比或产生 git 工作区变更。

In [12]:
W8A8_FILE = RECIPE_ROOT / 'module/quantization/compressed_tensors/compressed_tensors_w8a8_int8.py'
W8A8_FILE.write_text(W8A8_ORIGINAL_CODE, encoding='utf-8')
print('[恢复] 已恢复原始 W8A8 源码，baseline 对比不受污染')

[恢复] 已恢复原始 W8A8 源码，baseline 对比不受污染


## 8 提交要求

> 完整的提交说明参考[启航营课程实践说明](https://gitcode.com/cann/cann-launch-camp/blob/master/2026/University/HIT/First-session/README.md)。

1. 提交完整的 Jupyter Notebook（.ipynb），**保留所有单元格输出**。
2. 提交前依次运行所有单元格，确保：
   - 自定义算子编译成功；
   - 算子功能测试通过；
   - 性能数据收集成功。
3. **文件命名**：`gitcode账号_result.ipynb`。
4. **允许并且鼓励**使用 AI 工具辅助，但须独立思考并在报告中简述（解决了什么疑惑、是否引入新 bug）。
5. **不得抄袭。**

---
### 小结

本章完成了 QmmCustom 算子的开发、编译和单算子功能性能测试，并将该自定义算子接入了 Qwen3-8B 量化模型，验证功能并观测了其在模型中的性能。


### 附录
实践相关参考材料链接：
- [Ascend C 高阶api文档](https://gitcode.com/cann/asc-devkit/blob/master/docs/api/SIMD-API/%E9%AB%98%E9%98%B6API/%E9%AB%98%E9%98%B6API.md)
- [昇腾社区文档](https://www.hiascend.com/zh/document)
- [Ascend C 算子开发系列教程](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development)
- [Ascend C 算子功能调试](https://gitcode.com/cann/cann-learning-hub/tree/master/tutorials/ascendc_operator_development/07_Troubleshooting)
- [Matmul最佳实践样例](https://gitcode.com/cann/asc-devkit/blob/master/examples/01_simd_cpp_api/05_best_practices/01_matrix_compute/matmul_high_performance/README.md)
- [CANN内置量化矩阵乘算子实现](https://gitcode.com/cann/ops-nn/tree/master/matmul/quant_batch_matmul_v3)